# Training with Script Mode

In script mode, we write the training logic for our implementation. Sagemaker still manages the infrastructure.

`train.py` follows the SageMaker script-mode contract:
- Hyperparameters arrive as CLI args (`--max_depth`, `--eta`, ...)
- Data channel locations arrive as env vars (`SM_CHANNEL_TRAIN`, `SM_CHANNEL_VALIDATION`)
- The model must be written to `SM_MODEL_DIR` for SageMaker to package it as the job's output

1. Load the data from s3 (raw) 
2. basic pre processing 
3. Store in local and s3 
4. Model training (local) 
5. Model training (SM instance)

In [ ]:
import boto3
import sagemaker
import pandas as pd

## TODO: Update the following variables with your own values
REGION = ""
BUCKET_NAME = ""
EXECUTION_ROLE_ARN = ""

boto_session = boto3.Session(region_name=REGION)
sm_session = sagemaker.Session(boto_session=boto_session)

## Load data from S3 

We save the split locally so we can test with local mode first, then upload it to S3 for
the real training job.

In [ ]:
raw_s3_uri = "" # Set this to the S3 URI of your raw data file

df = pd.read_csv(raw_s3_uri)
print(f"Columns : {df.columns.to_list()}")

### Quick data checks

Checking the class split before deciding `scale_pos_weight` below.

In [ ]:
df['type'].value_counts()

In [ ]:
df['isFraud'].value_counts()

In [ ]:
# Sanity-check the legit:fraud ratio by hand -- this is roughly what scale_pos_weight
# should be set to below (773.7), computed properly from the actual train split.
6354407/8213

In [ ]:
from sklearn.model_selection import train_test_split


# Same feature set as the built-in-XGBoost run, minus the leaky balance columns entirely
# (not just origDelta/destDelta) -- step and isFlaggedFraud are dropped too since they add
# little signal here. type is one-hot encoded since train.py needs all-numeric columns.
cols_to_drop = ["step", "nameOrig", "nameDest","oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest","isFlaggedFraud"]
df_new = df.drop(columns=cols_to_drop)

# One hot encode the "type" column
df_new = pd.get_dummies(df_new, columns=["type"], prefix="type")

train, val = train_test_split(df_new, test_size=0.2, stratify=df_new["isFraud"], random_state=42)

# Local mode is for smoke-testing train.py itself, not for a real fit -- 5k rows trains in
# seconds and is enough to catch a broken script before paying for a full remote job.
df_split = df_new[:5000]
train_local, val_local = train_test_split(df_split, test_size=0.2, stratify=df_split["isFraud"], random_state=42)

train_local.to_csv("train_local.csv", index=False)
val_local.to_csv("validation_local.csv", index=False)

# Full split, written with a header -- train.py looks up columns by name (unlike the
# built-in-XGBoost notebook, which needs a headerless, target-first CSV).
train.to_csv("train.csv", index=False)
val.to_csv("validation.csv", index=False)
print("Train and validation datasets created and saved locally.")


train_uri = sm_session.upload_data("train.csv", bucket=BUCKET_NAME, key_prefix="processed/script_mode/train")
val_uri = sm_session.upload_data("validation.csv", bucket=BUCKET_NAME, key_prefix="processed/script_mode/val")
print("Train and validation datasets updated to S3 successfully.")
print(f"Training file at : {train_uri}")
print(f"Validation file at : {val_uri}")

scale_pos_weight = (train["isFraud"] == 0).sum() / (train["isFraud"] == 1).sum()
print(f"train: {train.shape}, validation: {val.shape}, scale_pos_weight: {scale_pos_weight:.1f}")

## Local setup execution

Set `instance_type="local"` 

It will run the exact same container/script combination **in a local Docker container on your machine**, using local file paths instead of S3. 
This is the fast iteration loop. A bug caught here costs seconds, the same bug caught in a real training job
costs minutes of provisioning time plus the instance-hour charge. 

**NOTE - Requires Docker running locally.**

In [ ]:
# Sanity check the role ARN is set before deploying local_estimator below
EXECUTION_ROLE_ARN

In [ ]:
from sagemaker.xgboost import XGBoost

# Provide "source_dir" if entry point is inside another folder, otherwise it will not be able to find the entry point script

local_estimator = XGBoost(
    entry_point="train.py",
    framework_version="1.7-1",
    role=EXECUTION_ROLE_ARN,
    instance_count=1,
    instance_type="local",
    hyperparameters={
        "max_depth": 3,
        "eta": 0.2,
        "num_round": 25,  # smaller for a fast local smoke test
        "scale_pos_weight": scale_pos_weight,
    },
)

local_estimator.fit({"train": "file://train_local.csv", "validation": "file://validation_local.csv"})

## Real training job (using sagemaker)

Once the local run succeeds, switch `instance_type` to a real one and point the channels at
S3 instead of local files — nothing else about `train.py` or the hyperparameters changes.

In [ ]:
# Confirm these are the S3 (not local) paths before launching the real job below
train_uri, val_uri

In [ ]:
# Sagemaker Instance Mode (remote) training job -- this is the real training job, run it after the local smoke test above passes
estimator = XGBoost(
    entry_point="train.py",
    framework_version="1.7-1",
    role=EXECUTION_ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{BUCKET_NAME}/models/script-mode/",
    sagemaker_session=sm_session,
    hyperparameters={
        "max_depth": 5,
        "eta": 0.2,
        "num_round": 100,
        "scale_pos_weight": scale_pos_weight,
    },
    use_spot_instances=True,
    max_run=3600,              # max training time in seconds
    max_wait=7200,             # must be >= max_run
)

# This sets the env vars.
# for train - variable is SM_CHANNEL_TRAIN
# for validation - variable is SM_CHANNEL_VALIDATION
# format - if key is "abc", env variable is set is SM_CHANNEL_<ABC>
estimator.fit({"train": train_uri, "validation": val_uri})